In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from dotenv import load_dotenv
import os

d:\Internship Projects\coding\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\USER\AppData\Local\Temp\ipykernel_10408\2038170058.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading documnet

In [63]:
loader = PyPDFLoader("D:/Internship Projects/coding/RAG_project/Mohit_persnal_info.pdf")
docs = loader.load()

Deviding the loaded data into chunks

In [67]:
splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [68]:
len(chunks)

22

In [69]:
print(chunks[0].page_content)

Personal Profile and Life Journey — Mohit Kumar
Gupta
1. Current Profile Summary
Name: Mohit Kumar Gupta
Born: February 7, 2003
Hometown: Jamshedpur , Jharkhand
Current Location: Indore, Madhya Pradesh
Education: B.Tech in Computer Science and Engineering
University: Medi-Caps University, Indore
Graduation: 2026
Current Status: Final-Year Student
Target Roles: AI/ML Engineer , Python Developer
Professional Identity: AI/ML Enthusiast
Primary Interests: AI, ML, Deep Learning, Computer Vision, Generative AI, RAG
Major Deep Learning Work: ResNet-18, CIFAR-10, MixUp, AugMix
Development: Python, Streamlit, Docker , Kubernetes, Cloud Technologies


importing gemini api key

In [70]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

Now creating embedding from gemini embedding model and stoaring that data into vector store

In [71]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [72]:
vector_store.index_to_docstore_id

{0: '9d778751-a0ce-4a7a-8e02-ac14cebab958',
 1: 'b1f2853e-2b7a-4743-aa16-001a7b0b4d63',
 2: 'acec73f9-927b-41ae-9217-7c22114c4467',
 3: '3959d2d3-2fd7-40ac-bb1a-5d88433f70ce',
 4: 'a74509da-bcb8-4aae-8ddc-520d60fd392d',
 5: 'b5dc0ca6-eb9c-4cb2-8efc-ef9141fa5c88',
 6: '3e297513-5173-40ff-a883-c23e311177b1',
 7: 'd84a9892-a454-45b4-84f7-e672f6cb379f',
 8: 'd9ee2a0d-2d3a-404f-89bc-95aef252ba7f',
 9: '39aa4f92-e62f-42c2-aecc-810785d8d13c',
 10: '903cf975-99cd-4d7b-abbf-9fb6eb7c66c4',
 11: '7a542404-bffb-486b-9441-7d31f75a7cb2',
 12: 'f376ec9b-1ce5-4531-ab34-c784c11144f6',
 13: '2512c8fd-5d52-4f97-a7d9-66ff21dc4df1',
 14: '475378b0-3582-460c-bdad-41f20e5a14fe',
 15: '3ec3f952-536d-4c30-915d-6aa4e8df01d5',
 16: '66adbf0b-ae33-4acb-a736-59b0aba20d53',
 17: 'd95392f6-f2e2-4522-a1e6-2e4d129c262b',
 18: '6972cdd8-a871-413b-ad7d-48306a817ce3',
 19: '282404cc-14af-4341-a873-e4d20e9f49e5',
 20: '0d0a731b-769f-48a7-800f-f19d61f44d3a',
 21: '095a1e64-7bb7-4aed-bfc7-3b89da12bf83'}

In [73]:
vector_store.get_by_ids(['0db09b76-7cdd-44af-ae60-c7fe8fe3b02a'])

[]

Retriver : Now using retriver to retrive the data from the vector store

In [74]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
    )

In [75]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000187D5879810>, search_kwargs={'k': 4})

In [76]:
result = retriever.invoke('what are the skills of mohit')

In [77]:
result

[Document(id='d84a9892-a454-45b4-84f7-e672f6cb379f', metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Personal Profile and Life Journey — Mohit Kumar Gupta', 'author': 'ChatGPT Canvas', 'source': 'D:/Internship Projects/coding/RAG_project/Mohit_persnal_info.pdf', 'total_pages': 9, 'page': 2, 'page_label': '3'}, page_content='His primary areas of interest include:\nArtificial Intelligence\nMachine Learning\nDeep Learning\nComputer Vision\nGenerative AI\nPython-based AI development\nData Augmentation\nNeural Network Architectures\nHis professional goal is to develop practical AI/ML solutions and eventually work as an AI/ML Engineer or\nPython Developer.\n8. Technical Skills and Competencies\nMohit has developed experience with a variety of programming languages, frameworks, libraries, and\ndevelopment tools.\nProgramming and Development\nPython\nJava\nC++\nSQL\nMySQL\n• \n• \n• \n• \n• \n• \n• \n• \n• \n• \n• \n• \n• \n3'),
 Document(id='475378

Now adding LLM so that will get a perfect organised answer

In [78]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.2
)

In [79]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided document.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [80]:
question          = "tell me about mohit"
retrieved_docs    = retriever.invoke(question)

In [81]:
retrieved_docs

[Document(id='6972cdd8-a871-413b-ad7d-48306a817ce3', metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Personal Profile and Life Journey — Mohit Kumar Gupta', 'author': 'ChatGPT Canvas', 'source': 'D:/Internship Projects/coding/RAG_project/Mohit_persnal_info.pdf', 'total_pages': 9, 'page': 7, 'page_label': '8'}, page_content="16. Personal Interests and Hobbies\nCulinary Arts\nHe enjoys experimenting with vegetarian cooking, especially:\nPaneer\nBroccoli\nMushrooms\nLiterature and Poetry\nHe has a strong interest in Hindi and Urdu poetry, appreciating poets such as:\nGopal Das Neeraj\nTehzeeb Hafi\n17. Personality and Approach to Learning\nMohit's journey reflects a practical, hands-on learning approach: learning concepts, implementing them,\ndebugging issues, and improving through iteration.\nHis interest in AI/ML has evolved from basic models to full-scale systems involving APIs, vector databases,\nretrieval systems, and deployment pipelines.

In [82]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [83]:
context_text

"16. Personal Interests and Hobbies\nCulinary Arts\nHe enjoys experimenting with vegetarian cooking, especially:\nPaneer\nBroccoli\nMushrooms\nLiterature and Poetry\nHe has a strong interest in Hindi and Urdu poetry, appreciating poets such as:\nGopal Das Neeraj\nTehzeeb Hafi\n17. Personality and Approach to Learning\nMohit's journey reflects a practical, hands-on learning approach: learning concepts, implementing them,\ndebugging issues, and improving through iteration.\nHis interest in AI/ML has evolved from basic models to full-scale systems involving APIs, vector databases,\nretrieval systems, and deployment pipelines.\n18. Current Professional Direction\n\nMoving from Jamshedpur to Indore represented an important transition in his life. It exposed him to a\nbroader academic environment, new technologies, professional opportunities, and a wider range of\nexperiences.\nHe is currently a final-year student, with graduation expected in 2026.\n7. Development of Interest in Artificial I

In [84]:
final_prompt = prompt.invoke(
    {
        "context": context_text,
        "question": question
    }
)

In [85]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided document.\n      If the context is insufficient, just say you don't know.\n\n      16. Personal Interests and Hobbies\nCulinary Arts\nHe enjoys experimenting with vegetarian cooking, especially:\nPaneer\nBroccoli\nMushrooms\nLiterature and Poetry\nHe has a strong interest in Hindi and Urdu poetry, appreciating poets such as:\nGopal Das Neeraj\nTehzeeb Hafi\n17. Personality and Approach to Learning\nMohit's journey reflects a practical, hands-on learning approach: learning concepts, implementing them,\ndebugging issues, and improving through iteration.\nHis interest in AI/ML has evolved from basic models to full-scale systems involving APIs, vector databases,\nretrieval systems, and deployment pipelines.\n18. Current Professional Direction\n\nMoving from Jamshedpur to Indore represented an important transition in his life. It exposed him to a\nbroader academic environment, new technologies,

Generation

In [86]:
answer = llm.invoke(final_prompt)
print(answer.content)

Mohit was born on February 7, 2003, in Jamshedpur, Jharkhand, where he spent his early years and received his foundational education. He later moved to Indore, Madhya Pradesh, which exposed him to a broader academic environment and new opportunities.

He is currently a final-year student, expected to graduate in 2026. During his engineering education, his interests shifted towards Artificial Intelligence and Machine Learning, moving from basic models to full-scale systems involving APIs, vector databases, retrieval systems, and deployment pipelines. He has a practical, hands-on learning approach, focusing on implementing concepts, debugging, and improving through iteration.

Mohit has pursued professional certifications, including NPTEL certifications in "Programming with Generative AI" and "Software Conceptual Design." He is actively exploring modern AI application development, particularly Retrieval-Augmented Generation (RAG) systems.

Beyond academics and technology, Mohit has perso

In [87]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [88]:
def merge_docs(retrived_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [89]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(merge_docs),
    'question': RunnablePassthrough()
})

In [90]:
parallel_chain.invoke('who is Mohit')

{'context': "16. Personal Interests and Hobbies\nCulinary Arts\nHe enjoys experimenting with vegetarian cooking, especially:\nPaneer\nBroccoli\nMushrooms\nLiterature and Poetry\nHe has a strong interest in Hindi and Urdu poetry, appreciating poets such as:\nGopal Das Neeraj\nTehzeeb Hafi\n17. Personality and Approach to Learning\nMohit's journey reflects a practical, hands-on learning approach: learning concepts, implementing them,\ndebugging issues, and improving through iteration.\nHis interest in AI/ML has evolved from basic models to full-scale systems involving APIs, vector databases,\nretrieval systems, and deployment pipelines.\n18. Current Professional Direction\n\nMoving from Jamshedpur to Indore represented an important transition in his life. It exposed him to a\nbroader academic environment, new technologies, professional opportunities, and a wider range of\nexperiences.\nHe is currently a final-year student, with graduation expected in 2026.\n7. Development of Interest in 

In [91]:
parser = StrOutputParser()

In [92]:
main_chain = parallel_chain | prompt | llm | parser

In [93]:
main_chain.invoke("tell me about mohit in breif")

'Mohit, born on February 7, 2003, in Jamshedpur, Jharkhand, is currently a final-year student in Indore, Madhya Pradesh, expected to graduate in 2026. His interests lie strongly in Artificial Intelligence and Machine Learning, where he applies a practical, hands-on learning approach, evolving from basic models to full-scale systems and actively exploring RAG systems. He holds NPTEL certifications in Programming with Generative AI and Software Conceptual Design. Outside of academics, Mohit enjoys experimenting with vegetarian cooking, particularly paneer, broccoli, and mushrooms, and has a strong interest in Hindi and Urdu poetry. His move from Jamshedpur to Indore marked a significant transition, broadening his academic and professional exposure.'